# 🚀 XGBoost — Guía Educativa aplicada a las 7 Ligas

Este notebook explica **qué es XGBoost**, **cómo lo integramos** al portal de predicción (LASSO + Random Forest + XGBoost + Stacking de 3 modelos) y **cómo leer los resultados** de los 7 campeonatos.

**Pipeline del proyecto:**
1. Datos point-in-time por partido (Elo, valor de plantilla, estadísticas avanzadas, altitud, asistencia…).
2. Split temporal estricto: train (≤2023) → cal (2024) → test (2025+), sin leakage.
3. 3 modelos base: LASSO L1 (regresión logística), Random Forest y **XGBoost**.
4. Stacking de 3 vías con pesos óptimos sobre la calibración (Nelder-Mead).
5. Simulación Monte Carlo (50.000 temporadas) para proyecciones de campeón, copas y descenso.

## 1. ¿Qué es XGBoost?

**XGBoost** (eXtreme Gradient Boosting) es un algoritmo de **gradient boosting sobre árboles de decisión**.

A diferencia de un Random Forest (que **entrena muchos árboles en paralelo** y promedia), el boosting entrena los árboles **en secuencia**: cada árbol nuevo intenta corregir los **errores** de los anteriores.

**La idea en una frase:** un árbol solo es débil; **miles de árboles que se corrigen mutuamente** forman un modelo muy fuerte.

### ¿Por qué ganó tantas competencias?
- **Regularización** integrada (L1/L2) → menos sobreajuste que un GBM clásico.
- Manejo nativo de datos dispersos y de **características categóricas/numéricas heterogéneas**.
- Búsqueda de splits con **gradiente de la pérdida** (no solo impureza de árbol).
- **Paralelización** (los histogramas de características se calculan en paralelo) y ejecución muy rápida.

**Analogía:** Random Forest = comité de expertos independientes que votan. XGBoost = un equipo que entrena revisando el examen anterior: cada miembro estudia los errores del anterior.

## 2. Configuración y Carga

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, accuracy_score
from xgboost import XGBClassifier

LIGA = 'eng'  # cambia a mex / bra / chile / arg / esp / bund para explorar otras ligas
print('XGBoost listo')

## 3. Los hiperparámetros (tuneados por liga con CV temporal)

Realizamos un **grid search con TimeSeriesSplit (4 folds)** sobre el train (2020-2023) en cada liga, eligiendo el combo con menor log-loss CV — exactamente el mismo protocolo que usa el LASSO para elegir su C.

**XGBoost — el óptimo fue consistente en las 7 ligas (modelo pequeño y regularizado):**
```python
XGBClassifier(
    n_estimators=100,      # antes 250 (sobreadaptaba)
    max_depth=3,           # antes 4
    learning_rate=0.02,    # antes 0.05 (paso más fino)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)
```

**Random Forest — óptimos por liga** (más regularizado que el fijo inicial 200/5/15):
| Liga | max_depth | n_estimators | min_samples_split |
|---|---|---|---|
| eng | 5 | 500 | 30 |
| bund | 3 | 100 | 5 |
| esp | 3 | 500 | 30 |
| mex | 5 | 500 | 30 |
| bra | 3 | 500 | 30 |
| chile | 5 | 250 | 5 |
| arg | 3 | 100 | 30 |

**Lección:** en fútbol (datos pequeños y ruidosos), los hiperparámetros agresivos sobreajustan. El tuneo llevó a XGBoost de log-loss ~1.05 a ~0.97-1.02 según la liga — ahora es competitivo con LASSO y RF, y en Alemania el stacking le da ~55% de peso.

## 4. Split temporal (sin trampa)

El modelo se entrena solo con el **pasado** y se evalúa solo con el **futuro**:

| Conjunto | Temporadas | Uso |
|---|---|---|
| Train | 2020–2023 | Entrenar los 3 modelos base |
| Cal | 2024 | Ajustar pesos del stacking |
| Test | 2025+ | **Evaluación out-of-sample** (lo que reporta el portal) |

In [ ]:
import sys, os
sys.path.insert(0, f'{LIGA}/')
import motor

# M = motor.cargar(use_cache=True)   # carga el modelo entrenado desde cache (rápido)
M = motor.cargar_y_entrenar()        # entrena desde cero (solo la primera vez)

df = M['df_dataset']
print('Shape del dataset:', df.shape)
print('Temporadas:', sorted(df['temporada'].unique()))
print()
print('Métricas Out-of-Sample (2025+):')
for k, v in M['metricas'].items():
    w = f"  w={v.get('w')}" if 'w' in v else ''
    print(f"  {k:9s} → Log-Loss {v['logloss']:.4f} | Acc {v['accuracy']:.1f}%{w}")

## 5. Entrenamiento de XGBoost paso a paso

Reproducimos exactamente lo que hace el motor de la liga: el modelo base se entrena en `train`, y para las métricas finales se re-entrena en `train + cal` (sin tocar el test).

In [ ]:
cols = M['features']
train_mask = df['temporada'] <= 2023
cal_mask = df['temporada'] == 2024
test_mask = df['temporada'] >= 2025

X_tr, y_tr = df.loc[train_mask, cols].fillna(0.0), df.loc[train_mask, 'resultado']
X_ca, y_ca = df.loc[cal_mask, cols].fillna(0.0), df.loc[cal_mask, 'resultado']
X_te, y_te = df.loc[test_mask, cols].fillna(0.0), df.loc[test_mask, 'resultado']

xgb = Pipeline([('sc', StandardScaler()),
                ('xgb', XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8,
                                      random_state=42, n_jobs=-1, eval_metric='mlogloss'))])
xgb.fit(X_tr, y_tr)

p = xgb.predict_proba(X_te)
print(f'XGBoost base → Log-Loss test {log_loss(y_te, p):.4f} | Acc {accuracy_score(y_te, p.argmax(axis=1)):.1%}')

## 6. Stacking de 3 modelos — el equipo completo

Cada modelo ve los datos de forma distinta:
- **LASSO L1** → bueno para señales lineales y robusto; descarta features irrelevantes automáticamente.
- **Random Forest** → captura interacciones no lineales.
- **XGBoost** → interacciones + regularización; a menudo el mejor *single model* en tabular.

El **stacking** aprende los pesos óptimos `w = [w_lasso, w_rf, w_xgb]` minimizando el log-loss sobre el **cal set** (que los modelos no vieron):

In [ ]:
from scipy.optimize import minimize

# Probabilidades de cada modelo base sobre cal y test
p_l = M['pipe_lasso'].predict_proba(X_ca)
p_r = M['pipe_rf'].predict_proba(X_ca)
p_x = xgb.predict_proba(X_ca)  # el xgb del motor es equivalente a este

def stk_loss(w):
    w = np.abs(w); w = w / w.sum()
    return log_loss(y_ca, np.clip(w[0]*p_l + w[1]*p_r + w[2]*p_x, 1e-7, 1-1e-7))

res = minimize(stk_loss, [0.4, 0.3, 0.3], method='Nelder-Mead')
w = np.abs(res.x); w = w / w.sum()
print('Pesos óptimos del stacking sobre cal:')
print(f'  LASSO={w[0]:.3f}  RF={w[1]:.3f}  XGB={w[2]:.3f}')

# Evaluación del ensemble en test
p_blend = (w[0]*M['pipe_lasso'].predict_proba(X_te) +
           w[1]*M['pipe_rf'].predict_proba(X_te) +
           w[2]*xgb.predict_proba(X_te))
print(f'Stacking 3 vías → Log-Loss test {log_loss(y_te, p_blend):.4f} | Acc {accuracy_score(y_te, p_blend.argmax(axis=1)):.1%}')

## 7. Importancia de variables en XGBoost

Los árboles cuentan **cuánto mejora** cada variable cuando se usa para partir los nodos (`gain`). A diferencia del LASSO (que da 0/1 por feature), XGBoost reparte la importancia entre todas.

In [ ]:
imp = pd.Series(xgb.named_steps['xgb'].feature_importances_, index=cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
imp.head(15).iloc[::-1].plot(kind='barh', color='#8b5cf6', ax=ax)
ax.set_title(f'Top-15 variables — XGBoost ({LIGA.upper()})')
ax.set_xlabel('Importancia (gain)')
plt.tight_layout(); plt.show()

## 8. Comparación de los 4 modelos por liga

Abrimos los motores de las 7 ligas (usa su cache de entrenamiento) y comparamos el log-loss out-of-sample.

In [ ]:
ligas = {'eng': 'Inglaterra', 'bund': 'Alemania', 'esp': 'España',
         'mex': 'México', 'bra': 'Brasil', 'chile': 'Chile', 'arg': 'Argentina'}

filas = []
for lg, nombre in ligas.items():
    sys.path.insert(0, f'{lg}/')
    try:
        mo = __import__(lg, fromlist=['motor']) if False else __import__('motor')
        mm = mo.cargar()
        for k in ['lasso', 'rf', 'xgb', 'stacking']:
            filas.append({'Liga': nombre, 'Modelo': k.upper(),
                          'Log-Loss': mm['metricas'][k]['logloss'],
                          'Acc %': mm['metricas'][k]['accuracy']})
    except Exception as e:
        print(f'{lg}: error {e}')
    finally:
        sys.path.pop(0)

resumen = pd.DataFrame(filas)
piv = resumen.pivot(index='Liga', columns='Modelo', values='Log-Loss')
piv.style.highlight_min(axis=1, color='lightgreen')

## 9. Proyecciones Monte Carlo con XGBoost

Las simulaciones de campeonato ya están precomputadas con **50.000 temporadas** por modelo en cada liga (se guardan en `simulacion_mc.pkl`). Cargamos la proyección del modelo XGBoost:

In [ ]:
import pickle

ruta = f'{LIGA}/simulacion_mc.pkl'
with open(ruta, 'rb') as fh:
    sim = pickle.load(fh)

proy = sim['results']['xgb']
proy.head(8)

## 10. ¿Cuándo ganó XGBoost? — Interpretación práctica

Al comparar ligas verás patrones típicos:
- En **plantillas dominadas por dinero** (Premier, Bundesliga, Brasil) la variable `squad_value_diff` manda; RF y stacking suelen quedar primeros.
- En **ligas parejas** (Argentina, México) XGBoost a veces supera al LASSO porque encuentra interacciones sutiles (p. ej. `altitude_diff × squad_value_diff`).
- El **stacking** rara vez pierde: al minimizar log-loss en cal con pesos libres, `w` se adapta (a veces `w_xgb ≈ 0` si no aporta en esa liga). **Que el stacking asigne peso 0 a un modelo no es un fallo**: es la selección de modelos del equipo funcionando.

### ¿Por qué usar 4 modelos en lugar de solo el mejor?
1. **Robustez:** el mejor modelo cambia según la liga y la temporada; el stacking se adapta solo.
2. **Diagnóstico:** comparar modelos delata leakage o features mal construidas.
3. **Consenso:** un ensemble diverso (lineal + árbol + boosting) es la receta ganadora estándar en tabular.

## 11. Para profundizar

- Ver la implementación real en `{liga}/motor.py` (`pipe_xgb_base`, `pipe_xgb`, `_stk3`, `stack_w`).
- Probar otros hiperparámetros: `max_depth=3`, `learning_rate=0.03`, `n_estimators=500` — observa el log-loss en cal y test.
- Explorar `eval_metric` alternativas y `early_stopping_rounds` para parar el entrenamiento cuando el validation set deja de mejorar.
- Probar la función `monte_carlo`/`simular_campeonato` con `modelo='xgb'` y `n_sims` menores para ver el efecto del ruido de simulación.